### I am trying to implement the vector db for the first time 

In [37]:
import pandas as pd

In [38]:
df = pd.read_csv(".//Dataset/Reviews.csv")

In [39]:
df.head(5)

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


### Keeping only review text

In [46]:
reviews = df["Text"].head(2000).tolist()

### Choosing an embedding model

In [47]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5942.82it/s]


### Generating Embeddings

In [48]:
embeddings = model.encode(
    reviews,
    show_progress_bar=True
)

Batches: 100%|██████████| 63/63 [00:10<00:00,  5.85it/s]


In [50]:
import chromadb

client = chromadb.Client()

# Reset to avoid duplicate/stale data while iterating
try:
    client.delete_collection("reviews")
except Exception:
    pass

collection = client.create_collection("reviews")

collection.add(
    ids=[str(i) for i in range(len(reviews))],
    embeddings=embeddings.tolist(),
    documents=reviews
)

In [51]:
results = collection.query(
    query_embeddings=query_vector.tolist(),
    n_results=5
)

In [56]:
query_embedding = model.encode(["The best energy bar"])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

print(results["documents"])

[["ENERGY BITES F'ING RIP YOUR FACE OFF WITH MOLTEN LAVA ENERGY.<br /><br />INFINITY ENERGY TO THE F'ING MAX.  I ATE A WHOLE BOX ONCE AND THREW A CAR AT A BABY.<br /><br />F'ING RAVE.<br /><br />SERIOUSLY THESE ARE GREAT.", 'Were just what my wife wanted and she loves them. Next she will get the other type of bar by the same vendor.', 'I bought these for my husband and he said they are the best energy shots out there. He takes one in the mornings and works hard all day. Good stuff!', "After years of drinking Red Bull, Rockstar, Monster and some others, I have found the holy grail of energy drinks. It kicks in within 2-3 minutes and it lasts an entire work day. After seven hours or so, you gradually come down. There is no anxiety or disphoria when it wears off. You just get very sleepy when you finally rest.<br /><br />Redline Power Rush is the very best product on the market. If it were illegal I would buy it on the black market. 'Nuff said.", "I'm a big fan of Outburst Energy Bites, a

In [ ]:
for doc, dist in zip(results["documents"][0], results["distances"][0]):
    print(f"{dist:.4f} - {doc[:100]}")

{'ids': [['1233', '1365', '289', '1245', '1232']], 'embeddings': None, 'documents': [["ENERGY BITES F'ING RIP YOUR FACE OFF WITH MOLTEN LAVA ENERGY.<br /><br />INFINITY ENERGY TO THE F'ING MAX.  I ATE A WHOLE BOX ONCE AND THREW A CAR AT A BABY.<br /><br />F'ING RAVE.<br /><br />SERIOUSLY THESE ARE GREAT.", 'Were just what my wife wanted and she loves them. Next she will get the other type of bar by the same vendor.', 'I bought these for my husband and he said they are the best energy shots out there. He takes one in the mornings and works hard all day. Good stuff!', "After years of drinking Red Bull, Rockstar, Monster and some others, I have found the holy grail of energy drinks. It kicks in within 2-3 minutes and it lasts an entire work day. After seven hours or so, you gradually come down. There is no anxiety or disphoria when it wears off. You just get very sleepy when you finally rest.<br /><br />Redline Power Rush is the very best product on the market. If it were illegal I would 